In [1]:
pip install tonic

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import time
import random
import csv
from dataclasses import dataclass
from typing import List, Tuple, Optional, Dict, Any, Union, Callable

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Dataset, Subset, random_split
import matplotlib.pyplot as plt
from tqdm import tqdm


# ==============================================================================
# 0.  Utilities
# ==============================================================================

def set_seed(seed: int = 42):
    random.seed(seed)
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def one_hot(y: torch.Tensor, num_classes: int) -> torch.Tensor:
    return F.one_hot(y.long(), num_classes=num_classes).float()


def default_device() -> str:
    return "cuda" if torch.cuda.is_available() else "cpu"


# ==============================================================================
# 1.  Few-shot sampler
# ==============================================================================

def few_shot_subset(
    dataset,
    k_shot: int,
    num_classes: int,
    seed: int = 42,
    label_attr: str = "targets",
) -> Subset:
    rng = random.Random(seed)
    try:
        labels = getattr(dataset, label_attr)
        if isinstance(labels, torch.Tensor):
            labels = labels.tolist()
        elif not isinstance(labels, list):
            labels = list(labels)
    except AttributeError:
        labels = [int(dataset[i][1]) for i in range(len(dataset))]

    per_class: Dict[int, List[int]] = {c: [] for c in range(num_classes)}
    for idx, lbl in enumerate(labels):
        per_class[int(lbl)].append(idx)

    selected = []
    for c in range(num_classes):
        pool = per_class[c]
        rng.shuffle(pool)
        chosen = pool[:k_shot]
        if len(chosen) < k_shot:
            raise ValueError(
                f"Class {c} has only {len(pool)} samples; "
                f"cannot satisfy k_shot={k_shot}."
            )
        selected.extend(chosen)

    rng.shuffle(selected)
    return Subset(dataset, selected)


# ==============================================================================
# 2.  Dataset loaders
# ==============================================================================

# ---------- 2a. Standard torchvision datasets (MNIST / FashionMNIST) ----------

def get_torchvision_loaders(
    dataset_name: str,
    root: str,
    batch_size: int,
    k_shot: Optional[int],
    num_classes: int,
    val_ratio: float = 0.1,
    seed: int = 42,
    device: str = "cpu",
):
    tfm = transforms.Compose([transforms.ToTensor()])
    ds = dataset_name.upper()

    if ds in ("FMNIST", "FASHIONMNIST"):
        TrainCls = datasets.FashionMNIST
        TestCls  = datasets.FashionMNIST
    elif ds == "KMNIST":
        TrainCls = datasets.KMNIST
        TestCls  = datasets.KMNIST
    else:
        TrainCls = datasets.MNIST
        TestCls  = datasets.MNIST

    train_full = TrainCls(root=root, train=True,  download=True, transform=tfm)
    test_ds    = TestCls (root=root, train=False, download=True, transform=tfm)

    if k_shot is not None:
        train_base = few_shot_subset(train_full, k_shot, num_classes, seed=seed)
    else:
        train_base = train_full

    n_total = len(train_base)
    n_val   = max(1, int(round(val_ratio * n_total)))
    n_train = n_total - n_val
    g = torch.Generator().manual_seed(seed)
    train_ds, val_ds = random_split(train_base, [n_train, n_val], generator=g)

    use_cuda    = str(device).startswith("cuda") and torch.cuda.is_available()
    num_workers = 2 if use_cuda else 0
    pin_memory  = use_cuda
    kw = dict(num_workers=num_workers, pin_memory=pin_memory,
              persistent_workers=(num_workers > 0))

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  **kw)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, **kw)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, **kw)
    return train_loader, val_loader, test_loader


# ---------- 2b. Caltech-101 (Binary Classification: Faces vs Motorbikes) -----

def get_caltech101_loaders(
    root: str,
    batch_size: int,
    k_shot: Optional[int],
    num_classes: int = 2,
    val_ratio: float = 0.10,
    test_ratio: float = 0.15,
    seed: int = 42,
    device: str = "cpu",
):
    tf = transforms.Compose([
        transforms.Resize((32, 32)),
        transforms.Grayscale(num_output_channels=1),
        transforms.ToTensor(),
        transforms.Lambda(lambda x: x.view(-1)),
    ])
    full_ds = datasets.Caltech101(root=root, download=True, transform=tf)

    cat_to_idx = {c: i for i, c in enumerate(full_ds.categories)}
    face_idx   = cat_to_idx["Faces_easy"]
    moto_idx   = cat_to_idx["Motorbikes"]

    indices = [i for i, lbl in enumerate(full_ds.y) if lbl in (face_idx, moto_idx)]
    label_map = {face_idx: 0, moto_idx: 1}

    class RemappedSubset(Dataset):
        def __init__(self, ds, idxs, lmap):
            self.ds = ds
            self.idxs = idxs
            self.lmap = lmap
            self.targets = [lmap[ds.y[i]] for i in idxs]

        def __len__(self):
            return len(self.idxs)

        def __getitem__(self, i):
            x, y = self.ds[self.idxs[i]]
            return x, self.lmap[y]

    binary_ds = RemappedSubset(full_ds, indices, label_map)

    n_total = len(binary_ds)
    n_test  = max(num_classes, int(round(test_ratio  * n_total)))
    n_val   = max(num_classes, int(round(val_ratio   * n_total)))
    n_train = n_total - n_test - n_val

    g = torch.Generator().manual_seed(seed)
    train_full_ds, val_ds, test_ds = random_split(
        binary_ds, [n_train, n_val, n_test], generator=g
    )

    if k_shot is not None:
        class _IndexedSubset(Dataset):
            def __init__(self, subset):
                self.subset  = subset
                self.targets = [subset.dataset.targets[i] for i in subset.indices]
            def __len__(self):        return len(self.subset)
            def __getitem__(self, i): return self.subset[i]

        wrapped  = _IndexedSubset(train_full_ds)
        train_ds = few_shot_subset(wrapped, k_shot, num_classes, seed=seed, label_attr="targets")
    else:
        train_ds = train_full_ds

    use_cuda    = str(device).startswith("cuda") and torch.cuda.is_available()
    num_workers = 2 if use_cuda else 0
    pin_memory  = use_cuda
    kw = dict(num_workers=num_workers, pin_memory=pin_memory,
              persistent_workers=(num_workers > 0))

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  **kw)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, **kw)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, **kw)
    return train_loader, val_loader, test_loader


# ---------- 2c. N-MNIST via Tonic ---------------------------------------------

def _nmnist_preprocess(x_raw, device):
    """x_raw: (B, T, 2, 34, 34) tonic tensor -> (B, T, 1156) merged frames."""
    x = x_raw.to(device).float()
    if x.dim() == 5:
        x = x.sum(dim=2)  # merge polarities
        x = x.view(x.size(0), x.size(1), -1)
    return x.clamp(0, 1)


def get_nmnist_loaders(
    root: str,
    batch_size: int,
    k_shot: Optional[int],
    num_classes: int = 10,
    val_ratio: float = 0.1,
    seed: int = 42,
    device: str = "cpu",
    n_time_bins: int = 10,
):
    import tonic
    import tonic.transforms as TT

    sensor_size = tonic.datasets.NMNIST.sensor_size
    frame_tf    = TT.ToFrame(sensor_size=sensor_size, n_time_bins=n_time_bins)
    tf          = tonic.transforms.Compose([frame_tf])

    full_train = tonic.datasets.NMNIST(save_to=root, train=True, transform=tf)
    test_ds    = tonic.datasets.NMNIST(save_to=root, train=False, transform=tf)

    # Build targets explicitly to support few_shot_subset efficiently
    full_train.targets = [int(full_train[i][1]) for i in range(len(full_train))]

    n_total = len(full_train)
    n_val   = max(1, int(round(val_ratio * n_total)))
    n_train = n_total - n_val
    g       = torch.Generator().manual_seed(seed)
    train_full_ds, val_ds = random_split(full_train, [n_train, n_val], generator=g)

    if k_shot is not None:
        class _IndexedSubset(Dataset):
            def __init__(self, subset):
                self.subset  = subset
                self.targets = [subset.dataset.targets[i] for i in subset.indices]
            def __len__(self):        return len(self.subset)
            def __getitem__(self, i): return self.subset[i]

        wrapped  = _IndexedSubset(train_full_ds)
        train_ds = few_shot_subset(wrapped, k_shot, num_classes, seed=seed, label_attr="targets")
    else:
        train_ds = train_full_ds

    def collate(batch):
        xs, ys = zip(*batch)
        return torch.from_numpy(np.stack(xs)), torch.tensor(ys)

    use_cuda    = str(device).startswith("cuda") and torch.cuda.is_available()
    num_workers = 2 if use_cuda else 0
    pin_memory  = use_cuda
    kw = dict(num_workers=num_workers, pin_memory=pin_memory,
              collate_fn=collate, persistent_workers=(num_workers > 0))

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  **kw)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, **kw)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, **kw)
    return train_loader, val_loader, test_loader


# ==============================================================================
# 3.  Metrics
# ==============================================================================

class MetricAccumulator:
    def __init__(self, num_classes: int, device: str = "cpu"):
        self.C = int(num_classes)
        self.device = device
        self.reset()

    def reset(self):
        self.tp      = torch.zeros(self.C, dtype=torch.long, device=self.device)
        self.fp      = torch.zeros(self.C, dtype=torch.long, device=self.device)
        self.fn      = torch.zeros(self.C, dtype=torch.long, device=self.device)
        self.correct = 0
        self.total   = 0

    @torch.no_grad()
    def update(self, pred: torch.Tensor, y: torch.Tensor):
        pred = pred.view(-1).long().clamp(0, self.C - 1)
        y    = y.view(-1).long().clamp(0, self.C - 1)
        self.total   += int(y.numel())
        self.correct += int((pred == y).sum().item())
        pred_count = torch.bincount(pred, minlength=self.C)
        true_count = torch.bincount(y,    minlength=self.C)
        tp = torch.bincount(pred[pred == y], minlength=self.C)
        self.tp += tp
        self.fp += pred_count - tp
        self.fn += true_count - tp

    @torch.no_grad()
    def compute(self, eps: float = 1e-8) -> Dict[str, float]:
        tp = self.tp.float();  fp = self.fp.float();  fn = self.fn.float()
        p  = tp / (tp + fp + eps)
        r  = tp / (tp + fn + eps)
        f1 = 2.0 * p * r / (p + r + eps)
        return {
            "acc":       float(self.correct / max(self.total, 1)),
            "precision": float(p.mean()),
            "recall":    float(r.mean()),
            "f1":        float(f1.mean()),
        }


def _pretty_metrics(m: Dict[str, float]) -> str:
    return (f"Acc {m['acc']:.4f} | P {m['precision']:.4f} "
            f"| R {m['recall']:.4f} | F1 {m['f1']:.4f}")


# ==============================================================================
# 4.  PC activation helpers
# ==============================================================================

def make_pc_activation(name: str):
    name = name.lower()
    if name == "sigmoid":
        f      = torch.sigmoid
        fprime = lambda z: torch.sigmoid(z) * (1 - torch.sigmoid(z))
        return f, fprime
    if name == "tanh":
        f      = torch.tanh
        fprime = lambda z: 1 - torch.tanh(z) ** 2
        return f, fprime
    if name == "relu":
        def f(z):      return torch.clamp(z, 0.0, 1.0)
        def fprime(z): return ((z > 0.0) & (z < 1.0)).float()
        return f, fprime
    return (lambda z: z), (lambda z: torch.ones_like(z))


# ==============================================================================
# 5.  LIF Neuron
# ==============================================================================

class LIFNeuron(nn.Module):
    def __init__(
        self,
        N: int,
        dt: float = 1.0,
        tau_m: float = 20.0,
        thr: float = 1.0,
        reset: float = 0.0,
        tau_ref: float = 2.0,
        device: str = "cpu",
    ):
        super().__init__()
        self.N       = N
        self.dt      = float(dt)
        self.tau_m   = float(tau_m)
        self.thr     = float(thr)
        self.reset   = float(reset)
        self.tau_ref = float(tau_ref)
        self.device  = device
        self.alpha   = float(np.exp(-dt / tau_m))
        self.reset_states(B=1)

    def reset_states(self, B: int):
        self.B          = B
        self.Vm         = torch.zeros(B, self.N, device=self.device)
        self.refr       = torch.zeros(B, self.N, device=self.device)
        self.refr_steps = max(1, int(round(self.tau_ref / self.dt)))

    @staticmethod
    def _surrogate(v: torch.Tensor, thr: float, slope: float = 10.0) -> torch.Tensor:
        return slope / (1.0 + slope * (v - thr).abs()) ** 2

    def forward(self, I: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        if I.shape[0] != self.B:
            self.reset_states(B=I.shape[0])

        can      = (self.refr <= 0).float()
        Vn       = self.alpha * self.Vm + (1.0 - self.alpha) * I
        spk_hard = ((Vn >= self.thr) & can.bool()).float()

        if self.training:
            spk_surr = self._surrogate(Vn, self.thr)
            spk = spk_hard + (spk_surr - spk_surr.detach())
        else:
            spk = spk_hard

        self.Vm = torch.where(
            spk_hard.bool(), torch.full_like(Vn, self.reset), Vn
        ).detach()
        self.refr = torch.where(
            spk_hard.bool(),
            torch.full_like(self.refr, float(self.refr_steps)),
            torch.clamp(self.refr - 1.0, min=0.0),
        ).detach()

        return spk, self.Vm


# ==============================================================================
# 6.  Hodgkin-Huxley Neuron
# ==============================================================================

class HHNeuron(nn.Module):
    class Gate:
        def __init__(self, B, N, device="cpu"):
            self.alpha = torch.zeros(B, N, device=device)
            self.beta  = torch.zeros(B, N, device=device)
            self.state = torch.zeros(B, N, device=device)

        def update(self, dt):
            a = self.alpha * (1.0 - self.state)
            b = self.beta  * self.state
            return torch.clamp(self.state + dt * (a - b), 0.0, 1.0)

        def set_inf(self):
            self.state = self.alpha / (self.alpha + self.beta + 1e-8)

    def __init__(self, N, dt=0.03, device="cpu", thr=10.0, reset=0.0, tau_ref=2.0):
        super().__init__()
        self.N       = N
        self.dt      = float(dt)
        self.device  = device
        self.thr     = float(thr)
        self.reset   = float(reset)
        self.tau_ref = float(tau_ref)

        self.ENa   = nn.Parameter(torch.tensor(115.0), requires_grad=False)
        self.EK    = nn.Parameter(torch.tensor(-12.0), requires_grad=False)
        self.Eleak = nn.Parameter(torch.tensor(10.6),  requires_grad=False)
        self.gNa   = nn.Parameter(torch.tensor(120.0), requires_grad=False)
        self.gK    = nn.Parameter(torch.tensor(36.0),  requires_grad=False)
        self.gLeak = nn.Parameter(torch.tensor(0.3),   requires_grad=False)
        self.Cm    = nn.Parameter(torch.tensor(1.0),   requires_grad=False)

        self.reset_states(B=1)

    def reset_states(self, B: int):
        dev      = self.device
        self.B   = B
        self.Vm  = torch.zeros(B, self.N, device=dev)
        self.m   = HHNeuron.Gate(B, self.N, dev)
        self.n   = HHNeuron.Gate(B, self.N, dev)
        self.h   = HHNeuron.Gate(B, self.N, dev)
        self._update_gates(self.Vm)
        self.m.set_inf();  self.n.set_inf();  self.h.set_inf()
        self.refr       = torch.zeros(B, self.N, device=dev)
        self.refr_steps = max(1, int(round(self.tau_ref / self.dt)))

    def _update_gates(self, V):
        V = torch.clamp(V, -100.0, 100.0)
        self.n.alpha = 0.01 * (10.0 - V) / (torch.exp((10.0 - V) / 10.0) - 1.0 + 1e-8)
        self.n.beta  = 0.125 * torch.exp(-V / 80.0)
        self.m.alpha = 0.1  * (25.0 - V) / (torch.exp((25.0 - V) / 10.0) - 1.0 + 1e-8)
        self.m.beta  = 4.0  * torch.exp(-V / 18.0)
        self.h.alpha = 0.07 * torch.exp(-V / 20.0)
        self.h.beta  = 1.0  / (torch.exp((30.0 - V) / 10.0) + 1.0)

    def _currents(self, V, I, m, n, h):
        INa = (m ** 3) * self.gNa * h * (V - self.ENa)
        IK  = (n ** 4) * self.gK      * (V - self.EK)
        Ile = self.gLeak               * (V - self.Eleak)
        return I - INa - IK - Ile

    def forward(self, I: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        if I.shape[0] != getattr(self, "B", None):
            self.reset_states(B=I.shape[0])

        self._update_gates(self.Vm)
        m = self.m.update(self.dt)
        n = self.n.update(self.dt)
        h = self.h.update(self.dt)

        dV = self._currents(self.Vm, I, m, n, h) / self.Cm
        Vn = self.Vm + self.dt * dV
        Vn = torch.tanh(Vn / 30.0) * 30.0

        can = (self.refr <= 0)
        spk = ((Vn >= self.thr) & can).float()
        self.Vm = torch.where(spk.bool(), torch.full_like(Vn, self.reset), Vn)
        self.m.state = m;  self.n.state = n;  self.h.state = h
        self.refr = torch.where(
            spk.bool(),
            torch.full_like(self.refr, float(self.refr_steps)),
            torch.clamp(self.refr - 1.0, min=0.0),
        )
        return spk, self.Vm


# ==============================================================================
# 7.  Generic PC-SNN backbone
# ==============================================================================

class PCSNNet(nn.Module):
    def __init__(
        self,
        layer_sizes: List[int],
        dt: float = 0.03,
        device: str = "cuda",
        current_gain: Union[float, List[float]] = 30.0,
        I_bias: Union[float, List[float]] = 2.0,
        thr: Union[float, List[float]] = 0.8,
        pc_activation: str = "relu",
        lr: float = 2e-4,
        weight_decay: float = 1e-4,
        input_encoding: str = "poisson",
        poisson_scale: float = 1.0,
        precompute_poisson: bool = True,
        neuron_type: str = "hh",
        neuron_kwargs: Optional[Dict] = None,
    ):
        super().__init__()
        assert len(layer_sizes) >= 2

        self.device  = torch.device(device)
        self.sizes   = layer_sizes
        self.dt      = float(dt)
        self.L       = len(layer_sizes) - 1
        self.S       = self.L

        self.input_encoding     = input_encoding.lower()
        self.poisson_scale      = float(poisson_scale)
        self.precompute_poisson = bool(precompute_poisson)
        assert self.input_encoding in ("poisson", "latency_first", "event_direct")

        self.f, self.fprime = make_pc_activation(pc_activation)

        self.syn = nn.ModuleList([
            nn.Linear(layer_sizes[i], layer_sizes[i + 1], bias=True)
            for i in range(self.L)
        ])
        for lin in self.syn:
            nn.init.xavier_uniform_(lin.weight, gain=0.5)
            nn.init.zeros_(lin.bias)

        def _as_list(v, n, name):
            if isinstance(v, (list, tuple)):
                if len(v) == 1:  return [float(v[0])] * n
                assert len(v) == n; return [float(x) for x in v]
            return [float(v)] * n

        self.current_gain = _as_list(current_gain, self.S, "current_gain")
        self.I_bias       = _as_list(I_bias,       self.S, "I_bias")
        self.thr_list     = _as_list(thr,           self.S, "thr")

        nkw         = neuron_kwargs or {}
        neuron_type = neuron_type.lower()
        if neuron_type == "lif":
            self.neurons = nn.ModuleList([
                LIFNeuron(N=layer_sizes[i + 1], dt=dt, thr=self.thr_list[i],
                          device=device, **nkw)
                for i in range(self.S)
            ])
        else:
            self.neurons = nn.ModuleList([
                HHNeuron(layer_sizes[i + 1], dt=dt, device=device,
                         thr=self.thr_list[i], **nkw)
                for i in range(self.S)
            ])

        self.opt = torch.optim.Adam(
            self.syn.parameters(), lr=lr, weight_decay=weight_decay
        )
        self._last_spike_sums: Optional[List[torch.Tensor]] = None

    @torch.no_grad()
    def forward_proxies(self, x_in_intensity: torch.Tensor, steps_spk: int) -> List[torch.Tensor]:
        if self.input_encoding == "event_direct":
            x_event = x_in_intensity.to(self.device).float().clamp(0, 1)
            T_avail = x_event.size(1)
            steps_use = min(T_avail, steps_spk)
            x0 = x_event[:, :steps_use, :].mean(dim=1).clamp(0, 1)
            B = x0.size(0)
        else:
            x0 = x_in_intensity.to(self.device).clamp(0, 1)
            steps_use = steps_spk
            B = x0.size(0)

        for cell in self.neurons:
            cell.reset_states(B)

        if self.input_encoding == "latency_first":
            max_time = float(steps_use)
            lat = (max_time * (1.0 - x0)).clamp(0.0, max_time)
            tgrid = torch.arange(1, steps_use + 1, device=self.device).view(1, 1, -1)
            input_spikes = (lat.unsqueeze(-1) <= tgrid).float()
            input_fired  = torch.zeros_like(x0, dtype=torch.bool, device=self.device)
            poisson_spikes = None
        elif self.input_encoding == "poisson":
            input_spikes = None
            input_fired  = None
            if self.precompute_poisson:
                p = (x0 * self.poisson_scale).clamp(0.0, 1.0)
                poisson_spikes = (
                    torch.rand(B, x0.size(1), steps_use, device=self.device) < p.unsqueeze(-1)
                ).float()
            else:
                poisson_spikes = None
        else: # event_direct
            input_spikes = None
            input_fired = None
            poisson_spikes = None

        spike_sums = [
            torch.zeros(B, self.sizes[i + 1], device=self.device)
            for i in range(self.S)
        ]

        for t in range(steps_use):
            if self.input_encoding == "event_direct":
                inp = x_event[:, t, :].clamp(0, 1)
            elif self.input_encoding == "poisson":
                if poisson_spikes is not None:
                    inp = poisson_spikes[:, :, t]
                else:
                    p = (x0 * self.poisson_scale).clamp(0, 1)
                    inp = (torch.rand_like(x0) < p).float()
            else: # latency_first
                inp = (input_spikes[:, :, t] * (~input_fired)).float()
                input_fired.logical_or_(inp.bool())

            r_prev = inp
            for i in range(self.S):
                h   = F.linear(r_prev, self.syn[i].weight, self.syn[i].bias)
                I   = h * self.current_gain[i] + self.I_bias[i]
                spk, _ = self.neurons[i](I)
                spike_sums[i] += spk
                r_prev = spk

        self._last_spike_sums = spike_sums
        proxies = [x0]
        for i in range(self.S):
            proxies.append((spike_sums[i] / float(steps_use)).clamp(0.0, 1.0))
        return proxies

    @torch.no_grad()
    def last_spike_sums(self):
        return self._last_spike_sums

    def pc_infer(self, x_init, y_target=None, T_infer=50, eta_x=0.05, clamp_output=True):
        L = self.L
        x = [xi.clone().detach().to(self.device) for xi in x_init]
        x[0] = x[0].clamp(0, 1)

        if clamp_output and (y_target is not None):
            x[L] = y_target.clone().detach().to(self.device).clamp(0, 1)

        z_cache = [None] * L

        for _ in range(T_infer):
            e    = [None] * (L + 1)
            e[0] = torch.zeros_like(x[0])
            for l in range(1, L + 1):
                idx          = l - 1
                z            = F.linear(x[l - 1], self.syn[idx].weight, self.syn[idx].bias)
                z_cache[idx] = z
                e[l]         = x[l] - self.f(z)
            for l in range(1, L):
                downstream = (e[l + 1] * self.fprime(z_cache[l])) @ self.syn[l].weight
                x[l]       = (x[l] - eta_x * (e[l] - downstream)).clamp_(0.0, 1.0)
            if not clamp_output:
                x[L] = (x[L] - eta_x * e[L]).clamp_(0.0, 1.0)

        with torch.no_grad():
            energy = 0.0
            for l in range(1, L + 1):
                idx    = l - 1
                z      = F.linear(x[l - 1], self.syn[idx].weight, self.syn[idx].bias)
                el     = x[l] - self.f(z)
                energy += 0.5 * (el ** 2).mean().item()

        return x, e, z_cache, energy

    def pc_learn(self, x, e, z_cache):
        L = self.L
        B = x[0].shape[0]
        self.opt.zero_grad()
        for idx in range(L):
            local = e[idx + 1] * self.fprime(z_cache[idx])
            self.syn[idx].weight.grad = -(local.T @ x[idx]) / B
            self.syn[idx].bias.grad   = -local.mean(dim=0)
        torch.nn.utils.clip_grad_norm_(self.syn.parameters(), max_norm=1.0)
        self.opt.step()

    def train_step(self, x_in_intensity, y_target, steps_spk, T_infer, eta_x):
        proxies = self.forward_proxies(x_in_intensity, steps_spk=steps_spk)
        x, e, z_cache, energy = self.pc_infer(
            proxies, y_target=y_target, T_infer=T_infer, eta_x=eta_x, clamp_output=True
        )
        self.pc_learn(x, e, z_cache)
        return energy, proxies


# ==============================================================================
# 8.  Evaluation helpers
# ==============================================================================

@torch.no_grad()
def spike_rate_epoch(
    model, 
    loader, 
    device, 
    steps_spk, 
    eval_seed=1234, 
    batch_preprocess_fn: Optional[Callable] = None
):
    model.eval()
    S            = model.S
    total_spikes = [0.0] * S
    total_denom  = [0.0] * S

    devs = ([torch.cuda.current_device()]
            if str(device).startswith("cuda") and torch.cuda.is_available() else [])
    with torch.random.fork_rng(devices=devs, enabled=True):
        torch.manual_seed(eval_seed)
        if devs: torch.cuda.manual_seed_all(eval_seed)
        for x, _y in loader:
            if batch_preprocess_fn is not None:
                x = batch_preprocess_fn(x, device)
            else:
                x = x.to(device, non_blocking=True).view(x.size(0), -1)

            B  = x.size(0)
            model.forward_proxies(x, steps_spk=steps_spk)
            ss = model.last_spike_sums()
            if ss is None: continue
            for li in range(S):
                total_spikes[li] += float(ss[li].sum())
                total_denom[li]  += float(B * ss[li].shape[1] * steps_spk)

    per_layer  = [total_spikes[li] / max(total_denom[li], 1.0) for li in range(S)]
    total_rate = sum(total_spikes) / max(sum(total_denom), 1.0)
    return {"per_layer": per_layer, "total": total_rate}


@torch.no_grad()
def eval_epoch(
    model, 
    loader, 
    device, 
    steps_spk, 
    T_infer_eval,
    eta_x_eval, 
    eval_mode="pc", 
    eval_seed=1234, 
    batch_preprocess_fn: Optional[Callable] = None
):
    model.eval()
    C        = model.sizes[-1]
    ff_accum = MetricAccumulator(C, device="cpu")
    pc_accum = MetricAccumulator(C, device="cpu")
    total_energy = 0.0
    total        = 0

    devs = ([torch.cuda.current_device()]
            if str(device).startswith("cuda") and torch.cuda.is_available() else [])
    with torch.random.fork_rng(devices=devs, enabled=True):
        torch.manual_seed(eval_seed)
        if devs: torch.cuda.manual_seed_all(eval_seed)
        for x, y in loader:
            if batch_preprocess_fn is not None:
                x = batch_preprocess_fn(x, device)
            else:
                x = x.to(device, non_blocking=True).view(x.size(0), -1)

            y = y.to(device, non_blocking=True)
            B = x.size(0)

            proxies = model.forward_proxies(x, steps_spk=steps_spk)
            ff_pred = proxies[-1].argmax(dim=1)
            ff_accum.update(ff_pred.cpu(), y.cpu())

            if eval_mode == "pc":
                x_settle, _, _, _ = model.pc_infer(
                    proxies, y_target=None, T_infer=T_infer_eval,
                    eta_x=eta_x_eval, clamp_output=False
                )
                pc_accum.update(x_settle[-1].argmax(dim=1).cpu(), y.cpu())
            else:
                pc_accum.update(ff_pred.cpu(), y.cpu())

            y_oh = one_hot(y, C)
            _, _, _, energy = model.pc_infer(
                proxies, y_target=y_oh, T_infer=T_infer_eval,
                eta_x=eta_x_eval, clamp_output=True
            )
            total_energy += float(energy) * B
            total        += B

    ff_m = ff_accum.compute()
    pc_m = pc_accum.compute()
    return {
        "ff_acc": ff_m["acc"],   "ff_precision": ff_m["precision"],
        "ff_recall": ff_m["recall"], "ff_f1": ff_m["f1"],
        "mode_acc": pc_m["acc"], "mode_precision": pc_m["precision"],
        "mode_recall": pc_m["recall"], "mode_f1": pc_m["f1"],
        "pc_energy": total_energy / max(total, 1),
    }


# ==============================================================================
# 9.  Configuration dataclass
# ==============================================================================

@dataclass
class Cfg:
    dataset:     str   = "MNIST"
    num_classes: int   = 10
    input_dim:   int   = 28 * 28

    hidden_size: int   = 512
    neuron_type: str   = "hh"

    steps_spk:      int   = 10
    input_encoding: str   = "poisson"
    poisson_scale:  float = 1.0

    current_gain: float = 30.0
    I_bias:       float = 2.0
    thr:          float = 0.8

    pc_activation:  str   = "relu"
    lr:             float = 2e-4
    weight_decay:   float = 1e-4
    T_infer_train:  int   = 100
    T_infer_eval:   int   = 50
    eta_x:          float = 0.05
    eval_mode:      str   = "pc"

    epochs:     int = 10
    batch_size: int = 64

    k_shot: Optional[int] = 5

    eval_seed: int = 1234
    ckpt:      str = "best_model.pt"

    @property
    def layer_sizes(self):
        return (self.input_dim, self.hidden_size, self.num_classes)


# ==============================================================================
# 10.  Training loop
# ==============================================================================

def train_model(
    model, 
    train_loader, 
    val_loader, 
    device, 
    cfg: Cfg, 
    batch_preprocess_fn: Optional[Callable] = None
):
    hist = {"train_energy": [], "train_mode": [], "val_mode": [],
            "val_energy": [], "val_spike_rate": [], "epoch_time": []}

    os.makedirs(os.path.dirname(cfg.ckpt) or ".", exist_ok=True)
    best_acc = -1.0

    print(f"\n===== TRAINING: {cfg.neuron_type.upper()}-PC | {cfg.dataset} "
          f"| k={cfg.k_shot} =====")

    for epoch in range(1, cfg.epochs + 1):
        model.train()
        t0         = time.time()
        energy_sum = 0.0
        total_seen = 0
        tr_accum   = MetricAccumulator(cfg.num_classes, device="cpu")

        pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{cfg.epochs}", ncols=110)
        for x_raw, y in pbar:
            if batch_preprocess_fn is not None:
                x = batch_preprocess_fn(x_raw, device)
            else:
                x = x_raw.to(device, non_blocking=True).view(x_raw.size(0), -1)

            y    = y.to(device, non_blocking=True)
            y_oh = one_hot(y, cfg.num_classes)

            energy, proxies = model.train_step(
                x, y_oh, steps_spk=cfg.steps_spk,
                T_infer=cfg.T_infer_train, eta_x=cfg.eta_x,
            )
            B = x.size(0)
            total_seen += B
            energy_sum += float(energy) * B

            if cfg.eval_mode == "pc":
                x_settle, _, _, _ = model.pc_infer(
                    proxies, y_target=None,
                    T_infer=max(1, cfg.T_infer_eval // 2),
                    eta_x=cfg.eta_x, clamp_output=False,
                )
                pred = x_settle[-1].argmax(dim=1)
            else:
                pred = proxies[-1].argmax(dim=1)

            tr_accum.update(pred.detach().cpu(), y.detach().cpu())
            m = tr_accum.compute()
            pbar.set_postfix({"E": f"{float(energy):.4f}", "TrAcc": f"{m['acc']:.3f}"})

        epoch_time   = time.time() - t0
        train_energy = energy_sum / max(total_seen, 1)
        train_mode_m = tr_accum.compute()

        val_stats = eval_epoch(
            model, val_loader, device,
            steps_spk=cfg.steps_spk, T_infer_eval=cfg.T_infer_eval,
            eta_x_eval=cfg.eta_x, eval_mode=cfg.eval_mode, eval_seed=cfg.eval_seed,
            batch_preprocess_fn=batch_preprocess_fn,
        )
        val_spike_rates = spike_rate_epoch(
            model, val_loader, device,
            steps_spk=cfg.steps_spk, eval_seed=cfg.eval_seed,
            batch_preprocess_fn=batch_preprocess_fn,
        )

        hist["train_energy"].append(train_energy)
        hist["train_mode"].append(train_mode_m)
        hist["val_mode"].append({k: val_stats[f"mode_{k}"]
                                 for k in ("acc", "precision", "recall", "f1")})
        hist["val_energy"].append(val_stats["pc_energy"])
        hist["val_spike_rate"].append(val_spike_rates)
        hist["epoch_time"].append(epoch_time)

        monitor = float(val_stats["mode_acc"])
        if monitor > best_acc:
            best_acc = monitor
            torch.save(model.state_dict(), cfg.ckpt)
            print(f"  [ckpt] best val acc={best_acc:.4f}")

        print(
            f"  Ep{epoch:02d} | E={train_energy:.5f} | "
            f"TR [{_pretty_metrics(train_mode_m)}] | "
            f"VAL [{_pretty_metrics(hist['val_mode'][-1])}] | "
            f"t={epoch_time:.1f}s"
        )

    if os.path.exists(cfg.ckpt):
        model.load_state_dict(torch.load(cfg.ckpt, map_location=device))
        print(f"  [restore] best checkpoint: {cfg.ckpt}")

    return hist


# ==============================================================================
# 11.  Single experiment runner
# ==============================================================================

def run_experiment(
    cfg: Cfg, 
    device: str,
    train_loader, 
    val_loader, 
    test_loader, 
    batch_preprocess_fn: Optional[Callable] = None
) -> Dict[str, Any]:
    set_seed(42)
    model = PCSNNet(
        layer_sizes        = list(cfg.layer_sizes),
        dt                 = 0.03,
        device             = device,
        current_gain       = cfg.current_gain,
        I_bias             = cfg.I_bias,
        thr                = cfg.thr,
        pc_activation      = cfg.pc_activation,
        lr                 = cfg.lr,
        weight_decay       = cfg.weight_decay,
        input_encoding     = cfg.input_encoding,
        poisson_scale      = cfg.poisson_scale,
        precompute_poisson = True,
        neuron_type        = cfg.neuron_type,
    ).to(device)

    hist = train_model(
        model, train_loader, val_loader, device, cfg, 
        batch_preprocess_fn=batch_preprocess_fn
    )

    test_stats = eval_epoch(
        model, test_loader, device,
        steps_spk=cfg.steps_spk, T_infer_eval=cfg.T_infer_eval,
        eta_x_eval=cfg.eta_x, eval_mode=cfg.eval_mode, eval_seed=cfg.eval_seed,
        batch_preprocess_fn=batch_preprocess_fn
    )
    test_rates = spike_rate_epoch(
        model, test_loader, device,
        steps_spk=cfg.steps_spk, eval_seed=cfg.eval_seed,
        batch_preprocess_fn=batch_preprocess_fn
    )

    result = {
        "dataset":        cfg.dataset,
        "neuron_type":    cfg.neuron_type,
        "k_shot":         cfg.k_shot,
        "test_acc":       test_stats["mode_acc"],
        "test_f1":        test_stats["mode_f1"],
        "test_energy":    test_stats["pc_energy"],
        "spike_rate":     test_rates["total"],
        "val_accs":       [m["acc"] for m in hist["val_mode"]],
        "train_energies": hist["train_energy"],
    }

    print(f"\n  *** TEST  Acc={result['test_acc']:.4f}  "
          f"F1={result['test_f1']:.4f}  "
          f"Energy={result['test_energy']:.5f}  "
          f"SpikeRate={result['spike_rate']:.5f} ***\n")
    return result


# ==============================================================================
# 12.  Plotting helpers
# ==============================================================================

def plot_results(all_results: List[Dict], save_dir: str = "."):
    os.makedirs(save_dir, exist_ok=True)

    datasets     = sorted({r["dataset"]     for r in all_results})
    neuron_types = sorted({r["neuron_type"] for r in all_results})

    for ds in datasets:
        ds_res  = [r for r in all_results if r["dataset"] == ds]
        ks_here = sorted({r["k_shot"] for r in ds_res if r["k_shot"] is not None})
        if not ks_here: continue

        x = np.arange(len(ks_here))
        w = 0.35
        fig, ax = plt.subplots(figsize=(8, 4))
        for j, nt in enumerate(neuron_types):
            accs = []
            for k in ks_here:
                match = [r for r in ds_res
                         if r["neuron_type"] == nt and r["k_shot"] == k]
                accs.append(match[0]["test_acc"] if match else 0.0)
            ax.bar(x + j * w - w / 2, accs, w, label=nt.upper())
        ax.set_xticks(x)
        ax.set_xticklabels([f"k={k}" for k in ks_here])
        ax.set_ylabel("Test Accuracy")
        ax.set_title(f"{ds} – Few-shot accuracy by neuron type")
        ax.legend();  ax.set_ylim(0, 1)
        fig.tight_layout()
        plt.savefig(os.path.join(save_dir, f"bar_{ds}.png"), dpi=120)
        plt.close(fig)

    for ds in datasets:
        fig, ax = plt.subplots(figsize=(8, 4))
        for r in all_results:
            if r["dataset"] != ds: continue
            ax.plot(range(1, len(r["val_accs"]) + 1), r["val_accs"],
                    label=f"{r['neuron_type'].upper()} k={r['k_shot']}")
        ax.set_xlabel("Epoch");  ax.set_ylabel("Val Accuracy")
        ax.set_title(f"{ds} – Validation accuracy curves")
        ax.legend(fontsize=7);  fig.tight_layout()
        plt.savefig(os.path.join(save_dir, f"curve_{ds}.png"), dpi=120)
        plt.close(fig)

    csv_path = os.path.join(save_dir, "summary.csv")
    with open(csv_path, "w", newline="") as f:
        writer = csv.DictWriter(
            f, fieldnames=["dataset", "neuron_type", "k_shot",
                           "test_acc", "test_f1", "test_energy", "spike_rate"]
        )
        writer.writeheader()
        for r in all_results:
            writer.writerow({k: r[k] for k in writer.fieldnames})
    print(f"\nSummary saved to {csv_path}")
    return csv_path


# ==============================================================================
# 13.  Main entry point
# ==============================================================================

if __name__ == "__main__":

    CALTECH101_ROOT = "./data/caltech101"
    NMNIST_ROOT     = "./data/nmnist_tonic"
    DATA_ROOT       = "./data"

    set_seed(42)
    torch.backends.cudnn.benchmark = True
    if torch.cuda.is_available():
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32        = True
        try:  torch.set_float32_matmul_precision("high")
        except Exception: pass

    device = default_device()
    print(f"Device: {device}")

    K_SHOTS      = [1, 5, 10, 20]
    NEURON_TYPES = ["hh", "lif"]

    BASE_EPOCHS    = 20
    BASE_BATCH     = 64
    BASE_HIDDEN    = 256
    BASE_STEPS_SPK = 10
    BASE_T_TRAIN   = 50
    BASE_T_EVAL    = 25

    MNIST_CFG = dict(
        dataset        = "MNIST",
        num_classes    = 10,
        input_dim      = 28 * 28,
        hidden_size    = BASE_HIDDEN,
        steps_spk      = BASE_STEPS_SPK,
        input_encoding = "poisson",
        poisson_scale  = 1.0,
        current_gain   = 30.0,
        I_bias         = 2.0,
        thr            = 0.8,
        pc_activation  = "relu",
        lr             = 2e-4,
        weight_decay   = 1e-4,
        T_infer_train  = BASE_T_TRAIN,
        T_infer_eval   = BASE_T_EVAL,
        eta_x          = 0.05,
        eval_mode      = "pc",
        epochs         = BASE_EPOCHS,
        batch_size     = BASE_BATCH,
        eval_seed      = 1234,
    )

    # FMNIST_CFG = dict(**MNIST_CFG, dataset="FASHIONMNIST")

    # Replaced to match binary Caltech stats analysis (1024 dim = 32x32, 2 classes)
    CALTECH_CFG = dict(
        dataset        = "CALTECH101",
        num_classes    = 2,
        input_dim      = 32 * 32,  # 1024
        hidden_size    = 512,
        steps_spk      = BASE_STEPS_SPK,
        input_encoding = "poisson",
        poisson_scale  = 1.0,
        current_gain   = 20.0,
        I_bias         = 1.5,
        thr            = 0.8,
        pc_activation  = "relu",
        lr             = 1e-4,
        weight_decay   = 1e-4,
        T_infer_train  = BASE_T_TRAIN,
        T_infer_eval   = BASE_T_EVAL,
        eta_x          = 0.03,
        eval_mode      = "pc",
        epochs         = BASE_EPOCHS,
        batch_size     = BASE_BATCH,
        eval_seed      = 1234,
    )

    # Replaced to match event_direct NMNIST stats analysis (1156 dim, event_direct)
    NMNIST_CFG = dict(
        dataset        = "NMNIST",
        num_classes    = 10,
        input_dim      = 1156,
        hidden_size    = BASE_HIDDEN,
        steps_spk      = BASE_STEPS_SPK,
        input_encoding = "event_direct",
        poisson_scale  = 1.0,
        current_gain   = 25.0,
        I_bias         = 2.0,
        thr            = 0.8,
        pc_activation  = "relu",
        lr             = 2e-4,
        weight_decay   = 1e-4,
        T_infer_train  = BASE_T_TRAIN,
        T_infer_eval   = BASE_T_EVAL,
        eta_x          = 0.05,
        eval_mode      = "pc",
        epochs         = BASE_EPOCHS,
        batch_size     = BASE_BATCH,
        eval_seed      = 1234,
    )

    all_results: List[Dict] = []

    for k_shot in K_SHOTS:
        for neuron_type in NEURON_TYPES:

            # # ----------------------------------------------------------
            # #  MNIST
            # # ----------------------------------------------------------
            # print(f"\n{'='*60}")
            # print(f"  MNIST | k={k_shot} | neuron={neuron_type}")
            # print(f"{'='*60}")
            # cfg = Cfg(**MNIST_CFG, k_shot=k_shot, neuron_type=neuron_type,
            #           ckpt=f"ckpt_MNIST_{neuron_type}_k{k_shot}.pt")
            # train_loader, val_loader, test_loader = get_torchvision_loaders(
            #     dataset_name="MNIST", root=DATA_ROOT,
            #     batch_size=cfg.batch_size, k_shot=cfg.k_shot,
            #     num_classes=cfg.num_classes, device=device,
            # )
            # all_results.append(run_experiment(cfg, device, train_loader, val_loader, test_loader))

            # # ----------------------------------------------------------
            # #  FashionMNIST
            # # ----------------------------------------------------------
            # print(f"\n{'='*60}")
            # print(f"  FashionMNIST | k={k_shot} | neuron={neuron_type}")
            # print(f"{'='*60}")
            # cfg = Cfg(**FMNIST_CFG, k_shot=k_shot, neuron_type=neuron_type,
            #           ckpt=f"ckpt_FMNIST_{neuron_type}_k{k_shot}.pt")
            # train_loader, val_loader, test_loader = get_torchvision_loaders(
            #     dataset_name="FASHIONMNIST", root=DATA_ROOT,
            #     batch_size=cfg.batch_size, k_shot=cfg.k_shot,
            #     num_classes=cfg.num_classes, device=device,
            # )
            # all_results.append(run_experiment(cfg, device, train_loader, val_loader, test_loader))

            # ----------------------------------------------------------
            #  Caltech-101 (Binary Classification setup)
            # ----------------------------------------------------------
            print(f"\n{'='*60}")
            print(f"  Caltech-101 (Binary) | k={k_shot} | neuron={neuron_type}")
            print(f"{'='*60}")
            cfg = Cfg(**CALTECH_CFG, k_shot=k_shot, neuron_type=neuron_type,
                      ckpt=f"ckpt_CALTECH_{neuron_type}_k{k_shot}.pt")
            train_loader, val_loader, test_loader = get_caltech101_loaders(
                root=CALTECH101_ROOT, batch_size=cfg.batch_size,
                k_shot=cfg.k_shot, num_classes=cfg.num_classes,
                device=device,
            )
            all_results.append(run_experiment(cfg, device, train_loader, val_loader, test_loader))

            # ----------------------------------------------------------
            #  N-MNIST  (Tonic - Event Direct)
            # ----------------------------------------------------------
            print(f"\n{'='*60}")
            print(f"  N-MNIST (Tonic) | k={k_shot} | neuron={neuron_type}")
            print(f"{'='*60}")
            cfg = Cfg(**NMNIST_CFG, k_shot=k_shot, neuron_type=neuron_type,
                      ckpt=f"ckpt_NMNIST_{neuron_type}_k{k_shot}.pt")
            train_loader, val_loader, test_loader = get_nmnist_loaders(
                root=NMNIST_ROOT, batch_size=cfg.batch_size,
                k_shot=cfg.k_shot, num_classes=cfg.num_classes,
                device=device, n_time_bins=cfg.steps_spk,
            )
            all_results.append(
                run_experiment(
                    cfg, device, train_loader, val_loader, test_loader,
                    batch_preprocess_fn=_nmnist_preprocess
                )
            )

    # ------------------------------------------------------------------
    #  Plots + CSV
    # ------------------------------------------------------------------
    plot_results(all_results, save_dir=".")

    print("\n" + "=" * 80)
    print(f"{'Dataset':>14} | {'Neuron':>6} | {'k':>4} | "
          f"{'Acc':>7} | {'F1':>7} | {'Energy':>10} | {'SpikeRate':>10}")
    print("=" * 80)
    for r in all_results:
        print(
            f"{r['dataset']:>14} | {r['neuron_type']:>6} | {str(r['k_shot']):>4} | "
            f"{r['test_acc']:>7.4f} | {r['test_f1']:>7.4f} | "
            f"{r['test_energy']:>10.5f} | {r['spike_rate']:>10.5f}"
        )
    print("=" * 80)
    print("\nAll plots and CSV saved.")

Device: cuda

  Caltech-101 (Binary) | k=1 | neuron=hh

===== TRAINING: HH-PC | CALTECH101 | k=1 =====


Epoch 1/20: 100%|████████████████████████████████████████| 1/1 [00:00<00:00,  2.24it/s, E=0.2516, TrAcc=0.500]


  [ckpt] best val acc=0.2358
  Ep01 | E=0.25156 | TR [Acc 0.5000 | P 0.2500 | R 0.5000 | F1 0.3333] | VAL [Acc 0.2358 | P 0.1179 | R 0.5000 | F1 0.1908] | t=0.4s


Epoch 2/20: 100%|████████████████████████████████████████| 1/1 [00:00<00:00, 15.91it/s, E=0.2512, TrAcc=0.500]


  Ep02 | E=0.25117 | TR [Acc 0.5000 | P 0.2500 | R 0.5000 | F1 0.3333] | VAL [Acc 0.2276 | P 0.1148 | R 0.4828 | F1 0.1854] | t=0.1s


...

  [ckpt] best val acc=0.6973
  Ep18 | E=0.01245 | TR [Acc 0.9350 | P 0.9413 | R 0.9350 | F1 0.9343] | VAL [Acc 0.6973 | P 0.7085 | R 0.6918 | F1 0.6797] | t=0.2s


Epoch 19/20: 100%|███████████████████████████████████████| 4/4 [00:00<00:00, 17.86it/s, E=0.0119, TrAcc=0.960]


  [ckpt] best val acc=0.7085
  Ep19 | E=0.01199 | TR [Acc 0.9600 | P 0.9636 | R 0.9600 | F1 0.9593] | VAL [Acc 0.7085 | P 0.7084 | R 0.7046 | F1 0.6964] | t=0.2s


Epoch 20/20: 100%|███████████████████████████████████████| 4/4 [00:00<00:00, 17.28it/s, E=0.0102, TrAcc=0.955]


  [ckpt] best val acc=0.7138
  Ep20 | E=0.01162 | TR [Acc 0.9550 | P 0.9593 | R 0.9550 | F1 0.9541] | VAL [Acc 0.7138 | P 0.7160 | R 0.7094 | F1 0.7017] | t=0.2s
  [restore] best checkpoint: ckpt_NMNIST_lif_k20.pt

  *** TEST  Acc=0.7258  F1=0.7146  Energy=0.01745  SpikeRate=0.00000 ***


Summary saved to ./summary.csv

       Dataset | Neuron |    k |     Acc |      F1 |     Energy |  SpikeRate
    CALTECH101 |     hh |    1 |  0.4162 |  0.3014 |    0.24471 |    0.03806
        NMNIST |     hh |    1 |  0.4325 |  0.3959 |    0.02549 |    0.04688
    CALTECH101 |    lif |    1 |  0.6054 |  0.4231 |    0.22140 |    0.00000
        NMNIST |    lif |    1 |  0.3984 |  0.3439 |    0.03024 |    0.00000
    CALTECH101 |     hh |    5 |  0.5243 |  0.4818 |    0.16125 |    0.01039
        NMNIST |     hh |    5 |  0.5891 |  0.5804 |    0.02048 |    0.05320
    CALTECH101 |    lif |    5 |  0.5838 |  0.3686 |    0.21774 |    0.00000
        NMNIST |    lif |    5 |  0.5905 |  0.5707 |    0.0247